# StyleMatch main-readiness diagnostic: ECoRe vs centroid

This notebook runs the frozen-encoder ECoRe evaluation on the Part 8.5 Gutenberg-expanded source-heldout artifact, then audits retrieval, false-return concentration, supported language/corpus subgroups, and the whole-author scorer cross-fit.

Important: Part 8.8 compares rankings on this same V3 query set. If Part 8.8 has already been viewed, this run is diagnostic only and cannot be the paper's new confirmatory test. It does not establish construct validity, public-benchmark generalization, or calibrated open-set rejection. The final gate deliberately stays NOT MAIN-READY unless those separate requirements are met.

Run top to bottom once for this diagnostic. Do not tune on these test results.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import numpy as np
import pandas as pd

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
EXP = REPO / 'artifacts/source_expansion_v2'
HELDOUT = EXP / 'gutenberg_targeted_v1/source_heldout_splits.parquet'
EMBEDDINGS = EXP / 'gutenberg_targeted_v1/frozen_encoder_eval'
OUT = EXP / 'main_readiness_ecore_diagnostic_v1'
print('repo:', REPO)
print('heldout:', HELDOUT)
print('embeddings:', EMBEDDINGS)
print('output:', OUT)

## Recover the exact evaluator without checking out the repo

The evaluator was added in commit 5b8ae4a, inside fdd1f98..f1be784. This cell fetches only that one file if it is absent. It does not switch branches, overwrite notebooks, or touch untracked files.

In [ ]:
EVALUATOR = REPO / 'scripts/evaluate_ecore_innovations.py'
if subprocess.run(['git', 'cat-file', '-e', 'f1be784^{commit}'], cwd=REPO, capture_output=True).returncode != 0:
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO, check=True)
expected = subprocess.check_output(
    ['git', 'show', 'f1be784:scripts/evaluate_ecore_innovations.py'], cwd=REPO
)
if EVALUATOR.exists() and EVALUATOR.read_bytes() != expected:
    raise RuntimeError('Evaluator exists but differs from f1be784; preserving it. Inspect before proceeding.')
if not EVALUATOR.exists():
    EVALUATOR.write_bytes(expected)
print('evaluator:', EVALUATOR)
print('sha256:', hashlib.sha256(EVALUATOR.read_bytes()).hexdigest())
print('commit containing file: 5b8ae4a; verified endpoint: f1be784')

## Input and leakage preflight

The evaluator needs the complete train/dev/test parquet and aligned Part 8.5 embeddings. This preflight checks schema, source separation, and row order before the test scores are computed.

In [ ]:
required = [
    HELDOUT,
    EMBEDDINGS / 'style_embedding_train_embeddings.npy',
    EMBEDDINGS / 'style_embedding_eval_embeddings.npy',
    EMBEDDINGS / 'style_embedding_scores.npz',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing frozen input(s):\n' + '\n'.join(missing))
if OUT.exists() and any(OUT.iterdir()):
    raise FileExistsError('Diagnostic output directory is non-empty; refusing to overwrite or silently rerun.')
frame = pd.read_parquet(HELDOUT)
needed = {'chunk_id', 'source_id', 'corpus', 'language', 'author_or_speaker', 'split'}
missing_columns = needed - set(frame.columns)
if missing_columns:
    raise ValueError('Missing columns: ' + str(sorted(missing_columns)))
if frame['chunk_id'].astype(str).duplicated().any():
    raise ValueError('chunk_id is not unique')
if not {'train', 'dev', 'test'}.issubset(set(frame['split'].astype(str))):
    raise ValueError('Expected train, dev, and test splits')
identity = frame['independent_source_id'] if 'independent_source_id' in frame else frame['source_id']
frame['_source_key'] = frame['corpus'].astype(str) + '::' + identity.fillna('').astype(str)
leakage = frame.groupby(['language', 'author_or_speaker', '_source_key'])['split'].nunique().gt(1)
if leakage.any():
    raise ValueError('Independent source appears in multiple splits')
print('rows:', len(frame), '| profiles:', frame[['language', 'author_or_speaker']].drop_duplicates().shape[0])
print('split counts:', frame['split'].value_counts().to_dict())
print('test independent sources:', frame.loc[frame['split'].eq('test'), '_source_key'].nunique())
print('evidence status: diagnostic reuse if Part 8.8 has been inspected')

## Run the fixed ECoRe test

No hyperparameter sweep is run here. The comparison of interest is the predeclared episodic author-heldout scorer against the same frozen representation's centroid.

In [ ]:
cmd = [
    sys.executable, 'scripts/evaluate_ecore_innovations.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--temperature', '0.08',
    '--author-folds', '5',
    '--hard-negatives', '12',
    '--bootstrap-runs', '5000',
    '--train-cap', '300',
    '--embedding-seed', '20260701',
    '--seed', '20260725',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)

## Main-method criteria on the diagnostic test

Exposure metrics reuse the project's source-balanced implementations. Subgroup uncertainty is bootstrapped by author-language profile, not by chunks. A subgroup with fewer than 10 profiles is marked underpowered rather than passed.

In [ ]:
from scripts.evaluate_hubness_reranking import exposure
from scripts.search_postwhitening_calibrators import source_balanced_maui

report = json.loads((OUT / 'ecore_innovation_metrics.json').read_text())
payload = np.load(OUT / 'ecore_innovation_scores.npz', allow_pickle=True)
profiles = payload['profiles'].astype(str)
labels = payload['y_true'].astype(int)
chunk_ids = payload['chunk_ids'].astype(str)
methods = {
    'centroid': payload['centroid'].astype(float),
    'ecore_author_heldout': payload['episodic_author_heldout'].astype(float),
}
lookup = frame.drop_duplicates('chunk_id').copy()
lookup['chunk_id'] = lookup['chunk_id'].astype(str)
aligned = lookup.set_index('chunk_id').loc[chunk_ids].reset_index()
if len(aligned) != len(labels):
    raise ValueError('Metadata and score rows do not align')
if not np.array_equal(aligned['split'].astype(str).to_numpy(), np.repeat('test', len(aligned))):
    raise ValueError('Score artifact contains unexpected non-test rows')
source_identity = aligned['independent_source_id'] if 'independent_source_id' in aligned else aligned['source_id']
groups = (aligned['corpus'].astype(str) + '::' + source_identity.fillna('').astype(str)).to_numpy()
query_languages = aligned['language'].astype(str).to_numpy()
profile_languages = np.asarray([profile.split('::', 1)[0] for profile in profiles])
for method in methods:
    methods[method][query_languages[:, None] != profile_languages[None, :]] = -np.inf

def row_values(scores, y):
    order = np.argsort(scores, axis=1)[:, ::-1]
    ranks = np.asarray([int(np.flatnonzero(row == true)[0]) + 1 for row, true in zip(order, y)])
    return {
        'mrr': 1.0 / ranks,
        'recall_at_1': (ranks <= 1).astype(float),
        'recall_at_3': (ranks <= 3).astype(float),
        'recall_at_5': (ranks <= 5).astype(float),
    }

def profile_means(values, y):
    return np.asarray([values[y == author].mean() for author in np.unique(y)])

def bootstrap_delta(base, challenger, y, metric, runs=5000, seed=20260916):
    b = profile_means(row_values(base, y)[metric], y)
    c = profile_means(row_values(challenger, y)[metric], y)
    delta = c - b
    rng = np.random.default_rng(seed)
    draws = delta[rng.integers(0, len(delta), size=(runs, len(delta)))].mean(axis=1)
    return {
        'delta': float(delta.mean()),
        'ci_low': float(np.quantile(draws, 0.025)),
        'ci_high': float(np.quantile(draws, 0.975)),
    }

def macro_table(scores, y):
    values = row_values(scores, y)
    return {name: float(profile_means(value, y).mean()) for name, value in values.items()}

metric_rows = []
for method, scores in methods.items():
    metric_rows.append({'method': method, **macro_table(scores, labels)})
metrics_table = pd.DataFrame(metric_rows).set_index('method')
print('Retrieval, macro over profiles')
display(metrics_table)

retrieval_deltas = {
    metric: bootstrap_delta(methods['centroid'], methods['ecore_author_heldout'], labels, metric, seed=20260916 + i)
    for i, metric in enumerate(['mrr', 'recall_at_1', 'recall_at_3', 'recall_at_5'])
}
print('Paired profile-bootstrap deltas: ECoRe minus centroid')
display(pd.DataFrame(retrieval_deltas).T)

exposure_rows = []
exposure_tables = {}
for method, scores in methods.items():
    concentration, author_exposure = exposure(scores, labels, profiles, groups=groups, top_k=3)
    maui = source_balanced_maui(scores, labels, query_languages, profile_languages, groups, top_k=3)
    exposure_rows.append({'method': method, **concentration, **maui})
    exposure_tables[method] = author_exposure
exposure_table = pd.DataFrame(exposure_rows).set_index('method')
print('False-return concentration and source-balanced MAUI@3')
display(exposure_table)

subgroups = []
for group_field in ['corpus', 'language']:
    for group_name in sorted(aligned[group_field].fillna('unknown').astype(str).unique()):
        rows = aligned[group_field].fillna('unknown').astype(str).eq(group_name).to_numpy()
        group_labels = labels[rows]
        n_profiles = len(np.unique(group_labels))
        result = {'dimension': group_field, 'group': group_name, 'n_profiles': n_profiles}
        if n_profiles < 10:
            result['status'] = 'UNDERPOWERED'
        else:
            result['mrr_delta'] = bootstrap_delta(methods['centroid'][rows], methods['ecore_author_heldout'][rows], group_labels, 'mrr', seed=20260930 + len(subgroups))
            result['recall_at_3_delta'] = bootstrap_delta(methods['centroid'][rows], methods['ecore_author_heldout'][rows], group_labels, 'recall_at_3', seed=20261030 + len(subgroups))
            result['status'] = 'PASS' if result['mrr_delta']['ci_low'] >= -0.02 and result['recall_at_3_delta']['ci_low'] >= -0.02 else 'FAIL'
        subgroups.append(result)
subgroup_report = pd.DataFrame([
    {
        'dimension': row['dimension'], 'group': row['group'], 'n_profiles': row['n_profiles'], 'status': row['status'],
        'mrr_delta': row.get('mrr_delta', {}).get('delta'), 'mrr_ci_low': row.get('mrr_delta', {}).get('ci_low'),
        'recall3_delta': row.get('recall_at_3_delta', {}).get('delta'), 'recall3_ci_low': row.get('recall_at_3_delta', {}).get('ci_low'),
    }
    for row in subgroups
])
print('Subgroup non-degradation; tolerance = -0.02 for MRR and Recall@3')
display(subgroup_report)

main_diagnostic = {
    'evidence_status': 'diagnostic_only_if_part_8_8_test_was_inspected',
    'test_input': str(HELDOUT),
    'evaluator_sha256': hashlib.sha256(EVALUATOR.read_bytes()).hexdigest(),
    'retrieval_deltas_ecore_minus_centroid': retrieval_deltas,
    'exposure_metrics': exposure_table.to_dict(orient='index'),
    'subgroups': subgroups,
    'ecore_direct_deployment_contrast': report['innovation_3_episodic_transfer']['direct_deployment_contrast_vs_centroid'],
    'author_heldout_scope': 'The scoring weights are cross-fitted across whole author-language profiles; candidate profiles still have train support and this is not unknown-author open-set evaluation.',
    'unmeasured_required_gates': ['independent human style-similarity judgments', 'two public benchmarks including domain shift', 'calibrated open-set rejection for ECoRe'],
}
(OUT / 'main_readiness_diagnostics.json').write_text(json.dumps(main_diagnostic, indent=2, ensure_ascii=False))
metrics_table.to_csv(OUT / 'main_readiness_retrieval.csv')
exposure_table.to_csv(OUT / 'main_readiness_exposure.csv')
subgroup_report.to_csv(OUT / 'main_readiness_subgroups.csv', index=False)
print('Wrote score-only audit outputs under:', OUT)

## Gate verdict

The retrieval gate uses the existing project non-inferiority margin of -0.01 for both MRR and Recall@3. Exposure must fall on all three prespecified quantities. Every adequately supported language/corpus group must satisfy the -0.02 subgroup margin. Even if these pass, this notebook cannot mark the project main-ready: open-set ECoRe, human style judgments, and two external benchmarks remain mandatory and are not produced here.

In [ ]:
mrr_ni = retrieval_deltas['mrr']['ci_low'] >= -0.01
r3_ni = retrieval_deltas['recall_at_3']['ci_low'] >= -0.01
exposure_falls = all(
    exposure_table.loc['ecore_author_heldout', metric] < exposure_table.loc['centroid', metric]
    for metric in ['source_balanced_maui_at_3', 'false_top3_gini', 'maximum_false_top3_share']
)
supported_groups = subgroup_report[subgroup_report['status'] != 'UNDERPOWERED']
subgroups_pass = len(supported_groups) > 0 and supported_groups['status'].eq('PASS').all()
core = {
    'MRR non-inferior (95% profile bootstrap, margin -0.01)': bool(mrr_ni),
    'Recall@3 non-inferior (95% profile bootstrap, margin -0.01)': bool(r3_ni),
    'MAUI@3, Gini, and max false-return share all lower': bool(exposure_falls),
    'Supported language/corpus subgroups non-degraded': bool(subgroups_pass),
    'Whole-author scorer cross-fit': 'measured; candidates remain known via train support',
    'ECoRe open-set calibration': 'NOT MEASURED',
    'Independent human style-similarity judgments': 'NOT MEASURED',
    'Two public external benchmarks incl. domain shift': 'NOT MEASURED',
}
print(pd.Series(core, name='status'))
mandatory_gates_complete = False  # human judgments, external benchmarks, and open-set evaluation remain absent
main_ready = bool(mrr_ni and r3_ni and exposure_falls and subgroups_pass and mandatory_gates_complete)
print('\nMAIN-READY:', 'YES' if main_ready else 'NO — evidence package is incomplete or a method gate failed')
print('Interpretation: diagnostic only if the Part 8.8 test was already opened; do not cite as confirmatory.')